# House Price PredictionA Linear Regression model that predicts house sale prices based on structural features (quality rating, living area, garage size, etc.), using the Ames Housing Dataset.**Dataset source:** [Ames Housing Dataset — Kaggle](https://www.kaggle.com/datasets/shashanknecrothapa/ames-housing-dataset)

## 1. Load the Dataset

In [ ]:
import pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport numpy as npdf = pd.read_csv('AmesHousing.csv')print(df.shape)df.head()

## 2. Exploratory Data AnalysisCheck structure, data types, and missing values before cleaning.

In [ ]:
df.info()

In [ ]:
df.isnull().sum().sort_values(ascending=False).head(20)

Most "missing" values in this dataset aren't actually missing data — they represent a house not having that feature at all (e.g. no pool, no garage, no basement). This calls for different fill strategies depending on what the column represents:- **Categorical column, missing = "doesn't have this feature"** → fill with `'None'`- **Numeric column, missing = "doesn't have this, so amount is 0"** → fill with `0`- **Numeric column, missing = genuine unknown** → fill with the `median`

## 3. Handle Missing Values

In [ ]:
# Rule 1: Categorical columns where missing = "doesn't have this feature"none_cols = ['Pool QC', 'Misc Feature', 'Alley', 'Fence', 'Fireplace Qu',             'Garage Type', 'Garage Finish', 'Garage Qual', 'Garage Cond',             'Bsmt Exposure', 'BsmtFin Type 2', 'Bsmt Qual', 'Bsmt Cond',             'BsmtFin Type 1', 'Mas Vnr Type']for col in none_cols:    df[col] = df[col].fillna('None')# Rule 2: Numeric columns where missing = "doesn't have this, so amount is 0"zero_cols = ['Garage Yr Blt', 'Mas Vnr Area', 'Bsmt Full Bath', 'Bsmt Half Bath']for col in zero_cols:    df[col] = df[col].fillna(0)# Rule 3: Numeric column with a genuine unknown gapdf['Lot Frontage'] = df['Lot Frontage'].fillna(df['Lot Frontage'].median())df.isnull().sum().sort_values(ascending=False).head(10)

In [ ]:
# Remaining columns with just 1 missing value each — safe to fill with median / modesmall_numeric_cols = ['Total Bsmt SF', 'Bsmt Unf SF', 'Garage Area', 'Garage Cars', 'BsmtFin SF 1', 'BsmtFin SF 2']for col in small_numeric_cols:    df[col] = df[col].fillna(df[col].median())# Electrical is categorical — fill with the most common value (mode)df['Electrical'] = df['Electrical'].fillna(df['Electrical'].mode()[0])# Final check — should be 0df.isnull().sum().sum()

## 4. Distribution of Sale Price

In [ ]:
plt.figure(figsize=(8,5))df['SalePrice'].hist(bins=40, color='seagreen', edgecolor='black')plt.title('Distribution of Sale Price')plt.xlabel('Sale Price')plt.ylabel('Number of Houses')plt.show()

**Observation:** Sale price is right-skewed — most houses fall between $100,000-$300,000, with a long tail of higher-priced homes extending out toward $700,000+.

## 5. Feature SelectionRather than using all 82 columns, 10 features were selected based on their expected relationship with price — a mix of quality, size, and age indicators.

In [ ]:
features = ['Overall Qual', 'Gr Liv Area', 'Garage Cars', 'Garage Area',            'Total Bsmt SF', '1st Flr SF', 'Full Bath', 'TotRms AbvGrd',            'Year Built', 'Year Remod/Add']X = df[features]y = df['SalePrice']X.head()

## 6. Correlation Analysis

In [ ]:
corr_data = X.copy()corr_data['SalePrice'] = ycorrelation = corr_data.corr()['SalePrice'].sort_values(ascending=False)correlation

In [ ]:
plt.figure(figsize=(10,8))sns.heatmap(corr_data.corr(), annot=True, cmap='coolwarm', fmt='.2f')plt.title('Feature Correlation with SalePrice')plt.tight_layout()plt.show()

**Observation:** `Overall Qual` (0.80) and `Gr Liv Area` (0.71) are the strongest predictors. The heatmap also reveals multicollinearity between some features — e.g. `Garage Cars` and `Garage Area` (0.89) — worth keeping in mind when interpreting coefficients later.

## 7. Train/Test Split and Model Training

In [ ]:
from sklearn.model_selection import train_test_splitfrom sklearn.linear_model import LinearRegressionX_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)model = LinearRegression()model.fit(X_train, y_train)print("Training set size:", X_train.shape)print("Testing set size:", X_test.shape)

## 8. Model Evaluation

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_scorey_pred = model.predict(X_test)mae = mean_absolute_error(y_test, y_pred)rmse = np.sqrt(mean_squared_error(y_test, y_pred))r2 = r2_score(y_test, y_pred)print("MAE:", mae)print("RMSE:", rmse)print("R² Score:", r2)

In [ ]:
plt.figure(figsize=(8,6))plt.scatter(y_test, y_pred, alpha=0.5)plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--')plt.xlabel('Actual Price')plt.ylabel('Predicted Price')plt.title('Actual vs Predicted House Prices')plt.show()

In [ ]:
residuals = y_test - y_predplt.figure(figsize=(8,6))plt.scatter(y_pred, residuals, alpha=0.5)plt.axhline(y=0, color='r', linestyle='--')plt.xlabel('Predicted Price')plt.ylabel('Residual (Actual - Predicted)')plt.title('Residual Plot')plt.show()

**Observation:** The model performs strongly for typical-priced homes ($100K-$300K), with predictions clustering tightly around the ideal line. For higher-priced homes (above ~$400K), the model consistently **underpredicts** — likely because premium homes have value drivers (location, luxury finishes) not captured by these 10 structural features. RMSE ($39,562) being notably higher than MAE ($24,856) confirms a small number of large misses are pulling up that metric, rather than errors being uniform across all houses.

## 9. Coefficient Analysis

In [ ]:
coefficients = pd.DataFrame({    'Feature': X.columns,    'Coefficient': model.coef_}).sort_values('Coefficient', ascending=False)coefficients

**Observation:** `Overall Qual` is the single strongest driver — each 1-point increase in quality rating adds ~$19,547 to predicted price. Interestingly, `TotRms AbvGrd` and `Full Bath` show *negative* coefficients. This isn't because more rooms/bathrooms hurt value — it's a sign of multicollinearity: these features are highly correlated with `Gr Liv Area`, which is already in the model. Once square footage is held constant, packing more rooms into the same footprint can signal smaller, more subdivided spaces.

## 10. Bonus: One-Hot EncodingDemonstrating categorical feature encoding using `Neighborhood`, and testing whether adding location data improves the model.

In [ ]:
df.select_dtypes(include='object').columns

In [ ]:
neighborhood_encoded = pd.get_dummies(df['Neighborhood'], prefix='Neighborhood', drop_first=True)df_encoded = pd.concat([df, neighborhood_encoded], axis=1)df_encoded = df_encoded.drop('Neighborhood', axis=1)df_encoded.info()

In [ ]:
features_with_neighborhood = features + list(neighborhood_encoded.columns)X2 = df_encoded[features_with_neighborhood]y2 = df_encoded['SalePrice']X2_train, X2_test, y2_train, y2_test = train_test_split(X2, y2, test_size=0.2, random_state=42)model2 = LinearRegression()model2.fit(X2_train, y2_train)y2_pred = model2.predict(X2_test)print("New R² Score (with Neighborhood):", r2_score(y2_test, y2_pred))print("Original R² Score (without Neighborhood): 0.8048")

**Observation:** Adding neighborhood/location data improved R² from 0.80 to 0.84 — confirming that location is a meaningful price driver independent of a house's physical characteristics.

## 11. Conclusion### Model PerformanceA Linear Regression model was trained on 10 features to predict house sale prices, using an 80/20 train-test split (2,344 training houses, 586 test houses).- **R² Score: 0.80** — the model explains 80% of the variation in house prices, a strong result for a simple linear model using only 10 of the 82 available features.- **MAE: $24,856** — on average, predictions are off by about $24,856.- **RMSE: $39,562** — notably higher than MAE, indicating a smaller number of houses are predicted with much larger errors.### What Drives Price the Most- **Overall Quality** is the single strongest driver — each 1-point increase adds ~$19,547.- **Garage capacity** matters significantly — each additional car space adds ~$7,473.- **Square footage** (living area, garage, basement, first floor) all show small positive per-square-foot effects, roughly $14-$59 depending on area type.### Multicollinearity Note`TotRms AbvGrd` and `Full Bath` showed negative coefficients due to their high correlation with `Gr Liv Area` — a reminder that coefficients in a multi-feature regression describe an "all else equal" relationship, not a standalone cause-and-effect claim.### Where the Model StrugglesThe model performs very well for typical-priced homes but consistently underpredicts homes above ~$400,000, likely due to premium value drivers (location, luxury finishes) not captured by these structural features alone.### Key Takeaways1. Overall Quality rating is, by a wide margin, the most valuable single predictor of price.2. A simple 10-feature Linear Regression model explains 80% of price variation — adding location data (via One-Hot Encoding) improved this to 84%.3. Multicollinearity between correlated features can produce counterintuitive coefficients — a reminder to interpret linear model coefficients carefully.4. Future improvement could include more features, non-linear models (Random Forest, Gradient Boosting), or removing redundant correlated features.